<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 3.2.1
# *Querying the International Space Station*

## The OpenNotify API

The OpenNotify API exposes a few attributes of the International Space Station (ISS) via a simple, authentication-free interface. The simplicity of this API precludes any need for a dedicated Python library. However, as with many APIs, it accepts requests according to HTTP standards and returns responses in JSON format, so the Python libraries request and json will make managing the I/O simpler still.

In [1]:
import requests
import json
from datetime import datetime, date, time

This request fetches the latest position of the international space station:

In [2]:
response = requests.get("http://api.open-notify.org/iss-now.json")

Print the status code and text of the response:

In [3]:
#ANSWER
response.status_code

200

In [4]:
#ANSWER
print(response.text)

{"iss_position": {"latitude": "35.8191", "longitude": "67.9746"}, "timestamp": 1744281786, "message": "success"}


We can use another API to request the current position of the ISS and the next few times at which it will be over a certain location. The latitude and longitude of Sydney are (-33.87, 151.21).

In [5]:
response = requests.get("https://api.g7vrd.co.uk/v1/satellite-passes/25544/-33.87/151.21.json?minelevation=0&hours=24")

Print the response header:

In [6]:
#ANSWER
print(response.headers)

{'Date': 'Thu, 10 Apr 2025 10:43:14 GMT', 'Server': 'Apache', 'X-Amzn-Trace-Id': 'Root=1-67f7a0c2-a2d8f32e68a68d23bb51551a;', 'Vary': 'Origin,Access-Control-Request-Method,Access-Control-Request-Headers', 'Access-Control-Allow-Origin': '*', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Cache-Control': 'no-cache, no-store, max-age=0, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'X-Frame-Options': 'DENY', 'Content-Type': 'application/json', 'Keep-Alive': 'timeout=5, max=100', 'Connection': 'Keep-Alive', 'Transfer-Encoding': 'chunked'}


Print the content of the response (the data that the server returned):

In [7]:
#ANSWER
print(response.content)

b'{"api_status":"ALPHA","request_timestamp":"2025-04-10T10:43:14.093763867Z","norad_id":25544,"satellite_name":"ISS","tle_last_retrieved":"2025-04-09T17:43:54.027403365Z","lat":-33.87,"lon":151.21,"hours":24,"min_elevation":0,"query_ms":13,"passes":[{"start":"2025-04-10T11:06:39.081Z","tca":"2025-04-10T11:11:09.081Z","end":"2025-04-10T11:16:09.081Z","aos_azimuth":267,"los_azimuth":148,"max_elevation":13.0},{"start":"2025-04-10T12:46:29.081Z","tca":"2025-04-10T12:49:29.081Z","end":"2025-04-10T12:52:44.081Z","aos_azimuth":222,"los_azimuth":152,"max_elevation":4.0},{"start":"2025-04-10T14:24:44.081Z","tca":"2025-04-10T14:28:14.081Z","end":"2025-04-10T14:31:34.081Z","aos_azimuth":204,"los_azimuth":127,"max_elevation":5.0},{"start":"2025-04-10T16:01:04.081Z","tca":"2025-04-10T16:06:04.081Z","end":"2025-04-10T16:11:09.081Z","aos_azimuth":213,"los_azimuth":80,"max_elevation":19.0},{"start":"2025-04-10T17:37:39.081Z","tca":"2025-04-10T17:43:09.081Z","end":"2025-04-10T17:48:24.081Z","aos_azimut

Note that this is a Python byte string:

In [8]:
print(type(response.content))

<class 'bytes'>


Print just the "content-type" value from the header:

In [9]:
#ANSWER
print(response.headers['content-type'])

application/json


JSON was designed to be easy for computers to read, not for people. The `requests` library can decode the JSON byte string:

In [10]:
overheads = response.json()
print(overheads)

{'api_status': 'ALPHA', 'request_timestamp': '2025-04-10T10:43:14.093763867Z', 'norad_id': 25544, 'satellite_name': 'ISS', 'tle_last_retrieved': '2025-04-09T17:43:54.027403365Z', 'lat': -33.87, 'lon': 151.21, 'hours': 24, 'min_elevation': 0, 'query_ms': 13, 'passes': [{'start': '2025-04-10T11:06:39.081Z', 'tca': '2025-04-10T11:11:09.081Z', 'end': '2025-04-10T11:16:09.081Z', 'aos_azimuth': 267, 'los_azimuth': 148, 'max_elevation': 13.0}, {'start': '2025-04-10T12:46:29.081Z', 'tca': '2025-04-10T12:49:29.081Z', 'end': '2025-04-10T12:52:44.081Z', 'aos_azimuth': 222, 'los_azimuth': 152, 'max_elevation': 4.0}, {'start': '2025-04-10T14:24:44.081Z', 'tca': '2025-04-10T14:28:14.081Z', 'end': '2025-04-10T14:31:34.081Z', 'aos_azimuth': 204, 'los_azimuth': 127, 'max_elevation': 5.0}, {'start': '2025-04-10T16:01:04.081Z', 'tca': '2025-04-10T16:06:04.081Z', 'end': '2025-04-10T16:11:09.081Z', 'aos_azimuth': 213, 'los_azimuth': 80, 'max_elevation': 19.0}, {'start': '2025-04-10T17:37:39.081Z', 'tca': '

What kind of object did this give us?

In [11]:
#ANSWER:
print(type(overheads))

<class 'dict'>


Python dicts are easier to work with, but the data we want is still buried in that data structure, so we have to dig it out. First, extract the `passes` value to a separate list:

In [12]:
#ANSWER:
passes = overheads['passes']

Now extract the `start` strings into an array called `srisetimes`:

In [13]:
#ANSWER:
srisetimes = [xpass['start'] for xpass in passes]

These are strings. We convert these to an array of Python `datetime` values called `risetimes`:

In [14]:
srisetimes = [datetime.strptime(xpass['start'], "%Y-%m-%dT%H:%M:%S.%fZ") for xpass in passes]

Finally, use `risetime.strftime` to print these in a format that people understand:

```
e.g.
18/10/22 07:05
18/10/22 08:41
18/10/22 10:20
18/10/22 12:00
18/10/22 01:37
18/10/22 03:13
```



In [15]:
#ANSWER:
risetimes = [datetime.strptime(xpass['start'], "%Y-%m-%dT%H:%M:%S.%fZ") for xpass in passes]
risetimes

[datetime.datetime(2025, 4, 10, 11, 6, 39, 81000),
 datetime.datetime(2025, 4, 10, 12, 46, 29, 81000),
 datetime.datetime(2025, 4, 10, 14, 24, 44, 81000),
 datetime.datetime(2025, 4, 10, 16, 1, 4, 81000),
 datetime.datetime(2025, 4, 10, 17, 37, 39, 81000),
 datetime.datetime(2025, 4, 10, 19, 17, 49, 81000),
 datetime.datetime(2025, 4, 11, 7, 9, 19, 81000),
 datetime.datetime(2025, 4, 11, 8, 41, 4, 81000),
 datetime.datetime(2025, 4, 11, 10, 18, 14, 81000),
 datetime.datetime(2025, 4, 11, 11, 57, 49, 81000)]

Finally, here is an endpoint that tells us who is on board:

In [16]:
response = requests.get("http://api.open-notify.org/astros.json")

Referring to the methods used above, extract the number of astronauts and their names:

In [17]:
#ANSWER:
astros = response.json()
print(astros)
print(astros["number"])
for astronaut in astros['people']:
    print(astronaut['name'])

{'people': [{'craft': 'ISS', 'name': 'Oleg Kononenko'}, {'craft': 'ISS', 'name': 'Nikolai Chub'}, {'craft': 'ISS', 'name': 'Tracy Caldwell Dyson'}, {'craft': 'ISS', 'name': 'Matthew Dominick'}, {'craft': 'ISS', 'name': 'Michael Barratt'}, {'craft': 'ISS', 'name': 'Jeanette Epps'}, {'craft': 'ISS', 'name': 'Alexander Grebenkin'}, {'craft': 'ISS', 'name': 'Butch Wilmore'}, {'craft': 'ISS', 'name': 'Sunita Williams'}, {'craft': 'Tiangong', 'name': 'Li Guangsu'}, {'craft': 'Tiangong', 'name': 'Li Cong'}, {'craft': 'Tiangong', 'name': 'Ye Guangfu'}], 'number': 12, 'message': 'success'}
12
Oleg Kononenko
Nikolai Chub
Tracy Caldwell Dyson
Matthew Dominick
Michael Barratt
Jeanette Epps
Alexander Grebenkin
Butch Wilmore
Sunita Williams
Li Guangsu
Li Cong
Ye Guangfu


## HOMEWORK


1. Write a simple handler for the response status code (refer to lab resources slide for HTTP response codes). As this Jupyter Notebook is an interactive device, the handler does not need to manage subsequent code execution (i.e. by branching or aborting execution), although it should return something that could be used to do so if deployed in a Python program.

In [18]:
#ANSWER:

def handleResponse(response, verbose=False):
    '''
    Returns Boolean Value, Status Code

    response: A response object (can be from requests or another HTTP library)
    verbose: A flag to print additional information (default is False)
    '''
    # if Status Code is 200 return false, and status code
    # Otherwise Return True and Status Code
    status_code = response.status_code

    # Check for successful response (200 OK)
    if status_code == 200:
        if verbose:
            print(f"Request was successful with status code: {status_code}")
        return False, status_code
    else:
        if verbose:
            print(f"Request failed with status code: {status_code}")
        return True, status_code


2. Test your response handler on some correct and incorrect API calls.

In [19]:
response = requests.get("http://api.open-notify.org/astros.json")
if handleResponse(response)[0]:
    print('API call failed. Resolve issue before continuing!')

response = requests.get("http://api.open-notify.org/iss-now.json")
handleResponse(response, True)[0]

Request was successful with status code: 200


False

>

>

>



---



---



> > > > > > > > > © 2025 Institute of Data


---



---



